# Aiscern — Audio AI Detector Fine-Tuning
**Platform:** Kaggle (T4 GPU)  
**Target:** `saghi776/aiscern-audio-detector`  
**Base model:** `facebook/wav2vec2-base`  
**Runtime:** ~90 minutes  
**Expected accuracy:** >84% on ASVspoof2019 test set (EER < 0.10)

### Before running:
1. Set Accelerator → **GPU T4 x2** in notebook settings
2. Add secret `HF_TOKEN` in Add-ons → Secrets (needs **write** permission)
3. Create `saghi776/aiscern-audio-detector` repo on huggingface.co/new

In [ ]:
# CELL 1 — Install dependencies
!pip install -q transformers==4.40.0 datasets accelerate huggingface_hub \
    librosa soundfile scikit-learn evaluate torch torchaudio

In [ ]:
# CELL 2 — Authenticate
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('✅ Authenticated with HuggingFace')

In [ ]:
# CELL 3 — Configuration
import torch

BASE_MODEL    = 'facebook/wav2vec2-base'
REPO_ID       = 'saghi776/aiscern-audio-detector'
SAMPLING_RATE = 16000   # wav2vec2 requires 16kHz
MAX_DURATION  = 5       # seconds — truncate/pad all to 5s
TARGET_LEN    = SAMPLING_RATE * MAX_DURATION  # 80,000 samples
BATCH_SIZE    = 8       # audio is memory-heavy
EPOCHS        = 4
LR            = 1e-4
SEED          = 42
MAX_SAMPLES   = 20000   # 10K real + 10K fake
OUTPUT_DIR    = '/kaggle/working/aiscern-audio-detector'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 4 — Load datasets
# Primary: ASVspoof2019 LA — gold standard for TTS/voice-clone detection
#   bonafide = real human = label 0
#   spoof    = TTS/voice-clone = label 1
# Secondary: WaveFake (GAN vocoders) + CommonVoice (real speech)

from datasets import load_dataset, concatenate_datasets, Audio
import numpy as np

all_splits = []

def normalise_label(val):
    s = str(val).lower().strip()
    if s in ('spoof','spoofed','fake','ai','1','generated','synthetic','label_1'): return 1
    if s in ('bonafide','genuine','real','human','0','label_0'):                   return 0
    return -1

# 1. ASVspoof2019
try:
    print('Loading ASVspoof2019 LA...')
    ds = load_dataset('DynamicSuperb/AudioDeepfakeDetection_ASVspoof2019LA',
                      split='train', token=HF_TOKEN, trust_remote_code=True)
    ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))
    lbl_col = next((c for c in ds.column_names if 'label' in c.lower()), None)
    if lbl_col:
        ds = ds.map(lambda x: {'label': normalise_label(x[lbl_col])})
        ds = ds.filter(lambda x: x['label'] != -1)
        ds = ds.select_columns(['audio', 'label'])
        all_splits.append(ds)
        real = ds.filter(lambda x: x['label']==0).num_rows
        fake = ds.filter(lambda x: x['label']==1).num_rows
        print(f'  ✅ ASVspoof2019: {len(ds):,} (real={real:,} fake={fake:,})')
except Exception as e:
    print(f'  ⚠️  ASVspoof2019 skipped: {e}')

# 2. WaveFake (GAN vocoders — all fake)
try:
    print('Loading WaveFake...')
    ds = load_dataset('balt0/WaveFake', split='train', token=HF_TOKEN)
    ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))
    ds = ds.map(lambda x: {'label': 1})
    ds = ds.select_columns(['audio', 'label'])
    ds = ds.shuffle(SEED).select(range(min(5000, len(ds))))
    all_splits.append(ds)
    print(f'  ✅ WaveFake: {len(ds):,} (all fake)')
except Exception as e:
    print(f'  ⚠️  WaveFake skipped: {e}')

# 3. Mozilla CommonVoice EN (real speech)
try:
    print('Loading CommonVoice EN...')
    ds = load_dataset('mozilla-foundation/common_voice_11_0', 'en',
                      split='train', token=HF_TOKEN, trust_remote_code=True)
    ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))
    ds = ds.map(lambda x: {'label': 0})
    ds = ds.select_columns(['audio', 'label'])
    ds = ds.shuffle(SEED).select(range(min(8000, len(ds))))
    all_splits.append(ds)
    print(f'  ✅ CommonVoice: {len(ds):,} (all real)')
except Exception as e:
    print(f'  ⚠️  CommonVoice skipped: {e}')

# 4. LibriSpeech (real speech backup)
try:
    print('Loading LibriSpeech...')
    ds = load_dataset('openslr/librispeech_asr', 'clean', split='train.100',
                      token=HF_TOKEN, trust_remote_code=True)
    ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLING_RATE))
    ds = ds.map(lambda x: {'label': 0})
    ds = ds.select_columns(['audio', 'label'])
    ds = ds.shuffle(SEED).select(range(min(5000, len(ds))))
    all_splits.append(ds)
    print(f'  ✅ LibriSpeech: {len(ds):,} (all real)')
except Exception as e:
    print(f'  ⚠️  LibriSpeech skipped: {e}')

if not all_splits:
    raise RuntimeError('No datasets loaded — check HF_TOKEN and internet access')

combined = concatenate_datasets(all_splits)
print(f'\nCombined: {len(combined):,} samples')

In [ ]:
# CELL 5 — Balance dataset and split
from datasets import concatenate_datasets

real_ds  = combined.filter(lambda x: x['label'] == 0).shuffle(SEED)
fake_ds  = combined.filter(lambda x: x['label'] == 1).shuffle(SEED)
n        = min(len(real_ds), len(fake_ds), MAX_SAMPLES // 2)

balanced = concatenate_datasets([
    real_ds.select(range(n)),
    fake_ds.select(range(n)),
]).shuffle(SEED)

split    = balanced.train_test_split(test_size=0.15, seed=SEED)
train_ds = split['train']
test_ds  = split['test']
val_split = train_ds.train_test_split(test_size=0.1, seed=SEED)
train_ds  = val_split['train']
val_ds    = val_split['test']

print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')
print(f'Balance — Real: {n:,}  Fake: {n:,}')

In [ ]:
# CELL 6 — Audio preprocessing
# wav2vec2 needs: raw 16kHz PCM, amplitude-normalised, padded/truncated to 5s
import numpy as np
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(BASE_MODEL)

def preprocess_audio(examples):
    processed = []
    for audio in examples['audio']:
        arr = np.array(audio['array'], dtype=np.float32)
        # Resample if needed
        if audio['sampling_rate'] != SAMPLING_RATE:
            import librosa
            arr = librosa.resample(arr, orig_sr=audio['sampling_rate'],
                                   target_sr=SAMPLING_RATE).astype(np.float32)
        # Amplitude normalise
        mx = np.max(np.abs(arr))
        if mx > 0:
            arr = arr / mx
        # Pad or truncate to TARGET_LEN
        if len(arr) < TARGET_LEN:
            arr = np.pad(arr, (0, TARGET_LEN - len(arr)))
        else:
            arr = arr[:TARGET_LEN]
        processed.append(arr)

    inputs = feature_extractor(
        processed,
        sampling_rate=SAMPLING_RATE,
        return_tensors='np',
        padding='max_length',
        max_length=TARGET_LEN,
        truncation=True,
    )
    return {
        'input_values': inputs['input_values'],
        'labels':       examples['label'],
    }

print('Preprocessing audio (this takes a few minutes)...')
train_ds = train_ds.map(preprocess_audio, batched=True, batch_size=32,
                         remove_columns=['audio'], desc='Train')
val_ds   = val_ds.map(preprocess_audio, batched=True, batch_size=32,
                       remove_columns=['audio'], desc='Val')
test_ds  = test_ds.map(preprocess_audio, batched=True, batch_size=32,
                        remove_columns=['audio'], desc='Test')

train_ds.set_format('torch')
val_ds.set_format('torch')
test_ds.set_format('torch')
print('✅ Audio preprocessing complete')

In [ ]:
# CELL 7 — Load wav2vec2 model
# id2label MUST be: 0='REAL', 1='FAKE'
# hf-analyze.ts matches: /fake|spoof|label_1|synthetic/i → FAKE  ✅
#                         /real|bonafide|label_0|human/i  → REAL  ✅
from transformers import Wav2Vec2ForSequenceClassification

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: 'REAL', 1: 'FAKE'},
    label2id={'REAL': 0, 'FAKE': 1},
)

# Freeze CNN feature extractor — only fine-tune transformer layers
# Prevents overfitting, speeds up training by ~40%
model.freeze_feature_encoder()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M params ({trainable/total*100:.0f}%)')

In [ ]:
# CELL 8 — Data collator for variable-length audio padding
from dataclasses import dataclass
from typing import Dict, List
import torch

@dataclass
class AudioDataCollator:
    feature_extractor: Wav2Vec2FeatureExtractor

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        input_values = [{'input_values': f['input_values']} for f in features]
        labels       = [f['labels'] for f in features]
        batch = self.feature_extractor.pad(
            input_values, return_tensors='pt', padding=True
        )
        batch['labels'] = torch.tensor(labels, dtype=torch.long)
        return batch

collator = AudioDataCollator(feature_extractor=feature_extractor)
print('✅ Data collator ready')

In [ ]:
# CELL 9 — Training arguments + metrics
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
import numpy as np
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

def compute_eer(labels, scores):
    """Equal Error Rate — lower is better. Standard ASVspoof metric."""
    fpr, tpr, _ = roc_curve(labels, scores)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[idx] + fnr[idx]) / 2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs > 0.5).astype(int)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, average='binary')),
        'auc':      float(roc_auc_score(labels, probs)),
        'eer':      compute_eer(labels, probs),
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    gradient_accumulation_steps=4,  # effective batch size = 32
    logging_steps=50,
    fp16=True,
    push_to_hub=True,
    hub_model_id=REPO_ID,
    hub_token=HF_TOKEN,
    hub_strategy='every_save',
    report_to='none',
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print('✅ Trainer ready')

In [ ]:
# CELL 10 — Train
print(f'Training on {len(train_ds):,} samples for {EPOCHS} epochs...')
print(f'Effective batch size: {BATCH_SIZE * 4}  LR: {LR}')
trainer.train()

In [ ]:
# CELL 11 — Evaluate on test set
print('Evaluating on test set...')
test_results = trainer.evaluate(test_ds)

print(f"\n{'='*40}")
print(f"TEST ACCURACY: {test_results['eval_accuracy']:.4f}  ({test_results['eval_accuracy']*100:.1f}%)")
print(f"TEST F1:       {test_results['eval_f1']:.4f}")
print(f"TEST AUC:      {test_results['eval_auc']:.4f}")
print(f"TEST EER:      {test_results['eval_eer']:.4f}  (lower is better)")
print(f"{'='*40}")

if test_results['eval_accuracy'] >= 0.84:
    print('✅ Target accuracy >84% achieved!')
if test_results['eval_eer'] <= 0.10:
    print('✅ Target EER ≤0.10 achieved!')

In [ ]:
# CELL 12 — Push to HuggingFace
commit_msg = (
    f"wav2vec2 TTS/voice-clone detection — "
    f"accuracy={test_results['eval_accuracy']:.3f} "
    f"f1={test_results['eval_f1']:.3f} "
    f"eer={test_results['eval_eer']:.3f}"
)
trainer.push_to_hub(commit_message=commit_msg)
feature_extractor.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f'\n✅ Model pushed to: https://huggingface.co/{REPO_ID}')

In [ ]:
# CELL 13 — Verify output format
# hf-analyze.ts expects: [{label: 'FAKE', score: X}, {label: 'REAL', score: Y}]
# 'FAKE' matches /fake/i  ✅
# 'REAL' matches /real/i  ✅
from transformers import pipeline
import numpy as np

print('Verifying inference output format...')
# Create a simple synthetic test tone (TTS-like: pure, smooth)
sample_rate = SAMPLING_RATE
duration    = 3  # seconds
t = np.linspace(0, duration, sample_rate * duration)
test_audio = (np.sin(2 * np.pi * 220 * t) * 0.5).astype(np.float32)

pipe   = pipeline('audio-classification', model=REPO_ID, token=HF_TOKEN,
                  device=0 if torch.cuda.is_available() else -1)
result = pipe({'array': test_audio, 'sampling_rate': sample_rate})
print(f'\nInference output:')
print(result)
print(f'\nExpected: [{{"label": "FAKE", "score": X}}, {{"label": "REAL", "score": Y}}]')

labels_out = [r['label'] for r in result]
assert 'FAKE' in labels_out, f'ERROR: FAKE label missing! Got: {labels_out}'
assert 'REAL' in labels_out, f'ERROR: REAL label missing! Got: {labels_out}'
print('\n✅ Output format correct — compatible with hf-analyze.ts')